# Transparency Portal

Brazilian municipal, state and federal governments must, by law, provide data on public spending, budget, and use of federal funds. Of one the tools used for that is "Portal da Transparência" (Transparency Portal), a site that provides such data for public access.

While every municipality, state and federal governments have their own individual Portal da Transparencia, the system behind it varies. Publicenter is a brazilian company that develops a "Portal da Transparencia" system that is used my many brazilian municipalities, including my hometown.

Due to this connection, this project aims to extract information from Publicent's Portal da Transparencia and ease its visualization.

## The study case

The chosen municipality is Lagoa Formosa, a small town located in the interior of the state of Minas Gerais and my hometown.

Lagoa Formosa's Portal da Transparencia can be accessed by this link: [Lagoa Formosa's Portal da Transparência](https://transparencia.lagoaformosa.mg.gov.br/#/transparencia)

This site offers data on many different aspects of public administration, such as, for instance, positions and wages, public contracts, statistical reports, etc., however this project will focus only on public expense data, which is divided budgetary and off-budgetary expenditure.

## The first step: Undestanding HTTP requests

We will begin this project by undestanding how the front-end interact with the system's back-end.
For this we will be using our browser dev tools to inspect HTTP requests and their responses. Lets start with the budgetary expenses:

<p align="center">
  <a href="documentation\images\00-first-header.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
  <a href="documentation\images\01-first-response.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
</p>

From this we can get the request url:
`https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina=20&pagina=1&termoBase64=&indAgrupamento=FOR&datInicio=2026-07-01&datFim=2026-07-31&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho=`

By analysing it we can notice some interesting query parameters, more specifically:
- `elementosPorPagina=` which decribes how many elements will be returned
- `pagina=`, which describes the page requested
- `datInicio=`, that is the starting date
- `datFim=`, that is the ending date

By analysing the response we found tree important keys:
- `"content"`, which is a list of entries
- `"totalElements"`, that is a number of the total number of data entries
- `"totalPages"`, that lists the number of pages
- `"size"`, that is the size of each page

With that we can begin playing with HTTP requests:


In [1]:
# we opted to use the httpx library instead of the requests library
import httpx

elementsPerPage = 20
page = 1
initialDate = '2010-01-01'
finalDate = '2026-07-29'

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40487, totalPages: 2025, size: 20


## The second step: Extracting data from Portal da Transparência

From this we can see that we have a total of 40487 entries distributed among 2025 pages.

We can play with the query parameters to try to reduce the amount of pages, and consequently reduce the number of requests we'll have to make to obtain all data.

Of course, we need to be careful to not trigger a timeout or any other block.

In [2]:
elementsPerPage = 10000

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40487, totalPages: 5, size: 10000


We can see that by setting the number of elements per page as 10000 we reduced the number of pages from 2025 to only 5.

There's not a magical number, so we always need to verify how many elements per page the systems support.

Anyway, we can now capture the data:

In [3]:
elementsPerPage = 10000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):

    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']
    
    page += 1

    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')




totalElements: 40487, elements obtained: 10000
totalElements: 40487, elements obtained: 20000
totalElements: 40487, elements obtained: 30000
totalElements: 40487, elements obtained: 40000
totalElements: 40487, elements obtained: 40487


## Third step: converting data for a tabular format

Now we have a list made of 40487 dictionaries, however it is still hard to work with data in this structure. For this reason we will convert it for a tabular format.

We could assume that every dictionary has the same keys to ease the extraction, but we'll check every entry to garantee no data is left behind.

To garantee we'll not need to request the data again, we'll also save it as an csv file, so we can open it latter if needed.

In [4]:
dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedBudgetaryData.append(row)

import csv
with open ('extractedBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedBudgetaryData)

## Doing the same for the off-budgetary expenditure.

We can notice that the request url is pretty similar: `https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina=20&pagina=1&termoBase64=&datInicio=2026-07-06&datFim=2026-07-31`

So we only need to work with the some parameters as before.

In [7]:
elementsPerPage = 1000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):
    
    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&datInicio={initialDate}&datFim={finalDate}'
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']    
    page += 1
    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')

dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedOffBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedOffBudgetaryData.append(row)

import csv
with open ('extractedOffBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedOffBudgetaryData)


totalElements: 2814, elements obtained: 1000
totalElements: 2814, elements obtained: 2000
totalElements: 2814, elements obtained: 2814


## Fourth step: Processing data
Now that we have extracted the data, we can work with it, and for this we have many options. 

In this project we'll use two: pandas and metabase:
- Pandas is a powerfull library focused on the manipulation, cleaning, and analysis of structured tabular data.
- Metabase is a user-friendly Business Intelligence (BI) tool that allows you to explore data, create charts, and build interactive dashboards, with or without SQL code.

In Pandas we'll be manipulating the data directly in this notebook, as for metabase, we'll convert the data into a sqlite database file and import it directly on metabase.

Now, lets generate both pandas' dataframes and sqlite files:

In [8]:
import pandas as pd
import sqlite3

budgetDf = pd.DataFrame(extractedBudgetaryData[1:], columns=extractedBudgetaryData[0])
offBudgetDf = pd.DataFrame(extractedOffBudgetaryData[1:], columns=extractedOffBudgetaryData[0])

# We'll be using pandas itself to connect to the sqlite and crete the tables:
dbconn = sqlite3.connect('portalDaTransparencia.sqlite')

budgetDf.to_sql(name='budgetData', con=dbconn, if_exists='replace', index=False)
offBudgetDf.to_sql(name='offBudgetData', con=dbconn, if_exists='replace', index=False)

dbconn.close()

print(f'length budgetDf: {len(budgetDf)}, length budgetDf: {len(offBudgetDf)}')

length budgetDf: 40487, length budgetDf: 2814
